In [23]:
# Imports
import tensorflow as tf
from tensorflow import keras

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from PIL import Image

In [17]:
# Load Model
model = keras.models.load_model(
    "../models/resnet50_pneumonia.keras"
)

In [18]:
# Load one image
img_path = "../data/CHEST X RAY/test/PNEUMONIA/person1_virus_9.jpeg"

In [19]:
# Read the image
img = Image.open(img_path)

img = img.convert("RGB")

img = img.resize((224,224))

img_array = np.array(img)

img_array = np.expand_dims(img_array, axis=0)

img_array = tf.keras.applications.resnet.preprocess_input(img_array)

In [20]:
# Check Prediction
prediction = model.predict(img_array)

prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step


array([[5.538064e-04, 9.994462e-01]], dtype=float32)

In [21]:
# Find the last convolution layer
for layer in model.layers:
    print(layer.name)

input_layer_5
sequential_1
resnet50
global_average_pooling2d_1
dropout_2
dense_2
dropout_3
dense_3


In [22]:
resnet_model = model.get_layer("resnet50")

for layer in resnet_model.layers:
    print(layer.name)

input_layer_4
conv1_pad
conv1_conv
conv1_bn
conv1_relu
pool1_pad
pool1_pool
conv2_block1_1_conv
conv2_block1_1_bn
conv2_block1_1_relu
conv2_block1_2_conv
conv2_block1_2_bn
conv2_block1_2_relu
conv2_block1_0_conv
conv2_block1_3_conv
conv2_block1_0_bn
conv2_block1_3_bn
conv2_block1_add
conv2_block1_out
conv2_block2_1_conv
conv2_block2_1_bn
conv2_block2_1_relu
conv2_block2_2_conv
conv2_block2_2_bn
conv2_block2_2_relu
conv2_block2_3_conv
conv2_block2_3_bn
conv2_block2_add
conv2_block2_out
conv2_block3_1_conv
conv2_block3_1_bn
conv2_block3_1_relu
conv2_block3_2_conv
conv2_block3_2_bn
conv2_block3_2_relu
conv2_block3_3_conv
conv2_block3_3_bn
conv2_block3_add
conv2_block3_out
conv3_block1_1_conv
conv3_block1_1_bn
conv3_block1_1_relu
conv3_block1_2_conv
conv3_block1_2_bn
conv3_block1_2_relu
conv3_block1_0_conv
conv3_block1_3_conv
conv3_block1_0_bn
conv3_block1_3_bn
conv3_block1_add
conv3_block1_out
conv3_block2_1_conv
conv3_block2_1_bn
conv3_block2_1_relu
conv3_block2_2_conv
conv3_block2_2_bn


In [24]:
# Get the last convolution layer
last_conv_layer_name = "conv5_block3_out"

In [25]:
# Create Prediction Func
def get_prediction(img_array):
    prediction = model.predict(img_array, verbose=0)

    predicted_class = np.argmax(prediction[0])

    return prediction, predicted_class

In [26]:
# Build the Grad-CAM model (Keras 3 version)
grad_model = tf.keras.Model(
    inputs=resnet_model.input,
    outputs=[
        resnet_model.get_layer(last_conv_layer_name).output,
        resnet_model.output
    ]
)

In [27]:
model.summary(expand_nested=True)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 224, 224,  │          0 │ input_layer_5[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ random_flip_1  │ (32, 224, 224, 3) │          0 │ -                 │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (32, 224, 224, 3) │          0 │ -                 │
│ random_rotation_1   │                   │            │                   │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ random_zoom_1  │ (32, 224, 224, 3) │          0 │ -                 │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ sequential_1[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ sequential_1[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ sequential_1[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_4  │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ conv1_pad      │ (None, 230, 230,  │          0 │ -                 │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ conv1_conv     │ (None, 112, 112,  │      9,472 │ -                 │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ conv1_bn       │ (None, 112, 112,  │        256 │ -                 │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ conv1_relu     │ (None, 112, 112,  │          0 │ -                 │
│ (Activation)        │ 64)               │            │                 

 Total params: 24,375,304 (92.98 MB)

 Trainable params: 262,530 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 525,062 (2.00 MB)